# Nemotron Remote Preflight

Run this notebook on your JupyterHub remote kernel.

## How we will use this notebook

- I will update this notebook for you whenever needed.
- You only run the cells I call out.
- If a cell fails, paste the output and I will patch the notebook again.

## Current flow

1. System + GPU sanity
2. **Config cell** (repo path + env vars)
3. Env var presence check
4. Repo discovery and git sanity
5. Grader tests
6. HF dataset download smoke
7. DeepSeek API smoke test

In [ ]:
import os
import sys
import platform
import subprocess


def sh(cmd: str):
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(p.stdout.strip() or "(no stdout)")
    if p.stderr.strip():
        print("STDERR:", p.stderr.strip())
    print("-" * 80)
    return p.returncode


print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("-" * 80)

_ = sh("nvidia-smi")

try:
    import torch

    print(
        "torch",
        torch.__version__,
        "cuda",
        torch.cuda.is_available(),
        "gpus",
        torch.cuda.device_count(),
    )
except Exception as e:
    print("torch import failed:", repr(e))

: 

In [ ]:
# --- CONFIG (edit this cell only) ---
# Set your remote Nemotron repo root.
# Example: /home/jovyan/work/Nemotron
NEMOTRON_REPO = "/workspace/nemotron"

# Git repo to clone into NEMOTRON_REPO if the code isn't mounted already.
GITHUB_REPO_URL = "https://github.com/bayntun/Nemotron-training.git"

# If true, will `git clone` into NEMOTRON_REPO when eval/test_grader.py is missing.
AUTO_CLONE_IF_MISSING = True

# If true, runs: pip install -U pip && pip install -r requirements.txt after cloning.
AUTO_INSTALL_DEPS = True

# Paste tokens here for this notebook session OR set them in JupyterHub env/secrets.
# Leave as "" if already set in the environment.
HF_TOKEN = ""
DEEPSEEK_API_KEY = ""

import os
from pathlib import Path

if NEMOTRON_REPO:
    os.environ["NEMOTRON_REPO"] = NEMOTRON_REPO

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if DEEPSEEK_API_KEY:
    os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY

print("Configured NEMOTRON_REPO:", os.getenv("NEMOTRON_REPO", "<unset>"))
print("Repo exists:", Path(os.getenv("NEMOTRON_REPO", "")).exists())
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))
print("DEEPSEEK_API_KEY set:", bool(os.getenv("DEEPSEEK_API_KEY")))

In [ ]:
missing = []
for k in ["HF_TOKEN", "DEEPSEEK_API_KEY"]:
    v = os.getenv(k, "")
    if not v:
        missing.append(k)
    print(k, "SET" if v else "MISSING", f"(len={len(v)})")

if missing:
    print("\nMissing env vars:", ", ".join(missing))
    print(
        "Set them in this notebook session or via JupyterHub secrets/env. "
        "(Do not paste values here; just ensure they are set.)"
    )

In [ ]:
from pathlib import Path


def find_repo_root() -> Path | None:
    # Optional override: set NEMOTRON_REPO to the Nemotron repo root.
    env_repo = os.getenv("NEMOTRON_REPO")
    if env_repo:
        p = Path(env_repo).expanduser()
        if (p / "eval" / "test_grader.py").exists():
            return p

    cwd = Path.cwd()

    candidates = [
        cwd,
        cwd / "Nemotron",
        Path("/home/jovyan/work/Nemotron"),
        Path("/home/jovyan/work"),
        Path("/home/jovyan/Nemotron"),
        Path.home() / "Nemotron",
    ]

    for p in candidates:
        if (p / "eval" / "test_grader.py").exists():
            return p

    return None


_ = sh("pwd")
_ = sh("ls -la")

repo_root = find_repo_root()
print("repo_root:", str(repo_root) if repo_root else "MISSING")

if repo_root is None:
    print("\nERROR: Nemotron repo not found.")
    print("Fix the CONFIG cell (NEMOTRON_REPO) or clone the repo to /home/jovyan/work/Nemotron.")
else:
    globals()["REPO_ROOT"] = repo_root
    _git = subprocess.run(
        ["git", "rev-parse", "--is-inside-work-tree"],
        cwd=str(repo_root),
        text=True,
        capture_output=True,
    )
    print("git inside repo:", (_git.stdout.strip() if _git.stdout else "?"))
    _head = subprocess.run(
        ["git", "log", "--oneline", "-1"],
        cwd=str(repo_root),
        text=True,
        capture_output=True,
    )
    print("git HEAD:", _head.stdout.strip() if _head.stdout else "<none>")


In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def ensure_repo_root() -> Path:
    if "REPO_ROOT" in globals() and REPO_ROOT is not None:
        return Path(REPO_ROOT)

    target = os.getenv("NEMOTRON_REPO", "")
    if not target:
        raise RuntimeError("REPO_ROOT missing and NEMOTRON_REPO is unset. Set CONFIG cell." )

    target_path = Path(target).expanduser()
    if (target_path / "eval" / "test_grader.py").exists():
        globals()["REPO_ROOT"] = target_path
        return target_path

    auto_clone = globals().get("AUTO_CLONE_IF_MISSING", True)
    github_url = globals().get("GITHUB_REPO_URL", "")
    if not auto_clone:
        raise RuntimeError(f"Repo missing at {target_path} and AUTO_CLONE_IF_MISSING=false")
    if not github_url:
        raise RuntimeError("Repo missing and GITHUB_REPO_URL is empty.")

    print(f"\nRepo missing at {target_path}. Cloning...")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    if target_path.exists():
        subprocess.run(["rm", "-rf", str(target_path)], check=False)

    subprocess.run(["git", "clone", github_url, str(target_path)], check=True)

    auto_install = globals().get("AUTO_INSTALL_DEPS", False)
    if auto_install:
        print("Installing requirements.txt...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-U", "pip"],
            check=True,
            text=True,
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-r",
                str(target_path / "requirements.txt"),
            ],
            check=True,
            text=True,
        )

    if not (target_path / "eval" / "test_grader.py").exists():
        raise RuntimeError("Clone completed but eval/test_grader.py still missing.")

    globals()["REPO_ROOT"] = target_path
    return target_path


repo_root = ensure_repo_root()
print("Using REPO_ROOT:", repo_root)

cmd = [sys.executable, "-m", "pytest", "eval/test_grader.py", "-q"]
p = subprocess.run(cmd, cwd=str(repo_root), text=True, capture_output=True)
print(p.stdout)
print(p.stderr)
print("exit_code:", p.returncode)

In [ ]:
import sys, subprocess

assert "REPO_ROOT" in globals(), "Run repo discovery cell first (it sets REPO_ROOT)."
assert os.getenv("HF_TOKEN"), "HF_TOKEN missing. Set it in CONFIG cell or JupyterHub env."

cmd = [sys.executable, "-m", "data.download", "--sft-only"]
p = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True, capture_output=True)
print(p.stdout[-4000:])
print(p.stderr[-2000:])
print("exit_code:", p.returncode)

In [ ]:
import sys, subprocess

assert "REPO_ROOT" in globals(), "Run repo discovery cell first (it sets REPO_ROOT)."
assert os.getenv("DEEPSEEK_API_KEY"), "DEEPSEEK_API_KEY missing. Set it in CONFIG cell or JupyterHub env."

cmd = [sys.executable, "-m", "teacher.smoke_test"]
p = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True, capture_output=True)
print(p.stdout[-4000:])
print(p.stderr[-2000:])
print("exit_code:", p.returncode)